## Step 3 — Processing Parameter Setup

This step defines the main processing parameters used throughout the Pléiades DSM generation workflow.

Because the workflow is executed mainly from Jupyter notebooks, the same parameters would otherwise need to be repeated in several Bash cells. To avoid duplication and reduce the risk of inconsistent settings, all important variables are stored in a dedicated Bash configuration file (`.sh`).

This configuration file contains the dataset name, working directory, image resolution, ASP stereo-correlation settings, stereo-matching kernel sizes, image identifiers, and output-folder structure.

For each site or acquisition date, a separate configuration file can be created. For example:

```text
StPaul_Aug24.sh
Merdaret_Aug24.sh
Merdaret_Oct24.sh
Berarde_Aug24.sh

### 3.1 Why Use a Bash Configuration File?

The `.sh` configuration file is used to centralize all parameters required by the DSM generation workflow.

Instead of manually redefining parameters in every Jupyter cell, the configuration file is sourced once and the variables become available to all subsequent Bash commands.

This approach has several advantages:

- it keeps the notebook cleaner and easier to read;
- it avoids repeating the same variables in many processing cells;
- it ensures that all ASP commands use the same parameter values;
- it makes it easy to switch between sites by changing only one file;
- it improves reproducibility because each DSM output can be linked to a specific configuration file.

In this workflow, the `.sh` file stores:

1. the base working directory;
2. the site or acquisition name;
3. the input image resolution;
4. the output DSM resolution;
5. the ASP stereo-correlation parameters;
6. the BM and MGM kernel sizes;
7. the stereo image identifiers;
8. the standard ASP output directories.

The general structure is:

```bash
#!/bin/bash

# 1. Base directory
# 2. Site name and resolution
# 3. Reference DEM path, if needed
# 4. Stereo-correlation settings
# 5. Algorithm-specific parameters
# 6. Image identifiers
# 7. Output directories


---

## Markdown block 3 — Parameter meaning

````markdown
### 3.2 Explanation of the Main Parameters

The configuration file is organized into several parameter groups.

#### Base Directory

```bash
BASE_DIR="/mnt/summer/USERS/KOAI/StPaul_Aug24"
cd "$BASE_DIR"
```

`BASE_DIR` defines the main working folder for the selected site or acquisition.  
The command `cd "$BASE_DIR"` moves the Bash session into this folder before running the ASP workflow.

This ensures that all relative paths used later in the notebook are interpreted from the correct project directory.

---

#### Site Name

```bash
area_name=StPaul_Aug
```

`area_name` is used as a short name for the dataset.  
It can be used later for naming output files, folders, logs, and DSM products.

For example, outputs can be labelled using the site and acquisition date:

```text
StPaul_Aug_BM_AB_CK15_SK25
```

This helps keep the results organized when several sites, stereo pairs, algorithms, and kernel sizes are tested.

---

#### Native Image Resolution

```bash
rawRes_meters=0.5
```

`rawRes_meters` defines the approximate native spatial resolution of the Pléiades panchromatic image.

For Pléiades, the panchromatic image resolution is approximately 0.5 m.

This value is used to define the output DSM resolution and to keep the workflow consistent across sites.

---

#### Conversion from Metres to Degrees

```bash
rawRes=$(awk -v rm="$rawRes_meters" 'BEGIN { printf "%.15f", 360 * rm / 40075000 }')
```

`rawRes` converts the image resolution from metres to geographic degrees.

This is useful when some ASP commands or map-projection steps require a resolution expressed in degrees.

The conversion is approximate and is based on the Earth circumference. It is mainly used here to define a consistent map-projection resolution before DSM generation.

---

#### Output Resolution

```bash
multiplier=2
resOut=$(echo $multiplier $rawRes | awk '{printf "%4.15f\n",$1*$2}')
resOut_meters=$(echo $multiplier $rawRes_meters | awk '{printf "%4.15f\n",$1*$2}')
```

`multiplier` defines how much the native resolution is scaled for the output product.

Here, the multiplier is set to `2`, meaning that the output DSM resolution is approximately:

```text
0.5 m × 2 = 1.0 m
```

`resOut` stores the output resolution in degrees, while `resOut_meters` stores the same resolution in metres.

In this workflow, a 1 m output DSM is used as a compromise between preserving high-resolution topographic information and reducing processing noise and computation time.

---

#### Reference DEM

```bash
#dem_name=LiDAR_DEMs/Merdaret_LiDAR_dsm2021_WGS84_31.tif
```

`dem_name` stores the path to the reference DEM.

The reference DEM can be used during map projection, alignment, and later DSM comparison.

In the example above, the line is commented using `#`, meaning that it is not active. If a reference DEM is required in later processing steps, this line should be activated and updated with the correct file path.

Example:

```bash
dem_name=LiDAR_DEMs/StPaul_LiDAR_dsm2021_UTM31N.tif
```

The reference DEM must be prepared carefully before use. It should have the same CRS and horizontal units as the ASP-derived DSMs.

---

#### Stereo-Correlation Memory and Tile Size

```bash
corr_mem_limit_mb=14336
corr_tile_size=6400
```

`corr_mem_limit_mb` defines the memory limit allowed for stereo correlation, in megabytes.

`corr_tile_size` defines the size of the processing tiles used during stereo correlation.

These values depend on:

- the size of the study area;
- the image dimensions;
- the available machine memory;
- the selected stereo algorithm.

Larger tile sizes can improve processing efficiency but require more memory. Smaller tile sizes reduce memory demand but may increase processing time.

---

#### Subpixel Refinement Mode

```bash
spr=2
```

`spr` defines the ASP subpixel refinement mode used after initial stereo correlation.

Subpixel refinement improves the disparity estimate beyond integer-pixel precision and can strongly influence final DSM quality.

In this workflow, the same subpixel refinement setting is kept consistent across the tested configurations, so that differences between DSMs mainly reflect the stereo pair, algorithm, and kernel-size settings.

---

#### Block Matching Parameters

```bash
bmCKernels=(15 25 35)
bmSKernels=(25 35 45)
bmCM=2
```

These variables define the Block Matching configurations.

`bmCKernels` contains the tested correlation kernel sizes.

`bmSKernels` contains the tested subpixel kernel sizes.

`bmCM` defines the correlation cost mode used for BM.

In this workflow, BM is tested with relatively large kernel sizes because local block matching depends strongly on the support window used for image matching. Larger kernels can stabilize matching in low-texture or complex terrain, but they may also smooth small topographic details.

---

#### MGM Parameters

```bash
mgmCKernels=(5 7 9)
mgmSKernels=(9 15 21)
mgmCM=4
```

These variables define the More Global Matching configurations.

`mgmCKernels` contains the tested MGM correlation kernel sizes.

`mgmSKernels` contains the tested MGM subpixel kernel sizes.

`mgmCM` defines the matching cost mode used for MGM.

MGM is tested with smaller correlation kernels than BM because MGM uses a more global optimization strategy. This allows smaller local windows while still improving disparity consistency across the image.

---

#### Image Identifiers

```bash
i=A
j=B
k=C
```

The variables `i`, `j`, and `k` define simplified identifiers for the three Pléiades images.

In this workflow:

```text
A = first image
B = second image
C = third image
```

These identifiers are used to automatically build stereo-pair names such as:

```text
AB
AC
BC
```

and tri-stereo configurations such as:

```text
ABC
CBA
BAC
```

This naming convention keeps the processing commands shorter and makes the output folders easier to interpret.

---

#### Output Directories

```bash
mkdir -p asp_logs/
mkdir -p asp_out/dems/
```

These commands create the main output folders if they do not already exist.

`asp_logs/` is used to store processing logs.

`asp_out/` is used to store ASP intermediate outputs.

`asp_out/dems/` is used to store the final DSM products.

The option `-p` prevents errors if the folders already exist.



### 3.3 Important Note: Reference DEM Alignment and Reprojection

Before using any reference elevation dataset, such as LiDAR, in this workflow, the coordinate reference system (CRS) must be checked carefully.

The reference DEM must match the CRS of the stereo-derived DSMs. If the reference DEM and the ASP outputs are not in the same CRS, projection mismatch can create artificial horizontal shifts and incorrect elevation differences.

---

#### Example from this workflow

In this workflow, the LiDAR DEM was originally provided in:

```text
Lambert-93 / RGF93
EPSG:2154
```

Lambert-93 is a standard projected coordinate system used in France.

However, the ASP outputs are generated in:

```text
WGS 84 / UTM Zone 31N
EPSG:32631
```

Therefore, the LiDAR DEM must be reprojected from Lambert-93 to UTM Zone 31N before being used for map projection, DSM alignment, co-registration, or DEM differencing.

---

#### Why this step is important

Reprojecting the reference DEM to the same CRS as the stereo-derived DSMs is necessary because it:

- ensures correct pixel alignment between the Pléiades DSM and the reference DEM;
- avoids horizontal projection mismatch;
- keeps the horizontal units consistent;
- prevents artificial elevation differences caused by spatial misalignment;
- enables reliable co-registration and DEM of Difference (DoD) analysis.

Even if both CRS use metre units, the DEMs still need to be in the same projection. For example, Lambert-93 and UTM Zone 31N are both projected CRS with metre units, but their coordinate grids are different.

---

#### How to determine the correct UTM zone

The UTM zone depends on the geographic location of the study area.

A simple way to identify it is to use the GeoPlaner UTM Zone Finder:

```text
https://www.geoplaner.com/
```

Steps:

1. Open the GeoPlaner website.
2. Enter the coordinates of the study area or click directly on the map.
3. Read the UTM zone value.
4. Use the corresponding UTM CRS in the reprojection command.

For southeast France, the correct UTM zone is commonly:

```text
UTM Zone 31N
EPSG:32631
```

---

#### Reprojecting the LiDAR DEM with GDAL

The LiDAR DEM can be reprojected using `gdalwarp`.

Example:

```bash
gdalwarp -t_srs "+proj=utm +zone=31 +datum=WGS84 +units=m +no_defs" \
  Alignment_DEMs/Merdaret_LiDAR_dtm2021.tif \
  Alignment_DEMs/Merdaret_LiDAR_dtm2021_UTM31N.tif
```

In this command:

- `-t_srs` defines the target spatial reference system;
- `+proj=utm` specifies the UTM projection;
- `+zone=31` defines UTM Zone 31;
- `+datum=WGS84` defines the WGS84 geodetic datum;
- `+units=m` ensures that the horizontal units are metres;
- the first `.tif` file is the original LiDAR DEM;
- the second `.tif` file is the reprojected output DEM.

After reprojection, the new DEM should be visually checked in QGIS together with the Pléiades images or ASP outputs to confirm that the spatial overlap is correct.

---

#### Updating the configuration file

After reprojection, the `.sh` configuration file should point to the reprojected DEM, not the original Lambert-93 DEM.

Example:

```bash
dem_name=Alignment_DEMs/Merdaret_LiDAR_dtm2021_UTM31N.tif
```

This ensures that all subsequent ASP processing steps use the correctly prepared reference DEM.

---

#### Useful references

- GDAL `gdalwarp` documentation: https://gdal.org/en/stable/programs/gdalwarp.html
- EPSG:2154 Lambert-93 / RGF93: https://epsg.io/2154
- EPSG:32631 WGS 84 / UTM Zone 31N: https://epsg.io/32631
- GeoPlaner UTM Zone Finder: https://www.geoplaner.com/

### 3.4 Loading the Configuration File in Jupyter

After preparing the `.sh` configuration file, it can be loaded inside a Jupyter Bash cell.

Example:

```bash
%%bash
source StPaul_Aug24.sh

echo "Loaded configuration for: $area_name"
echo "Base directory: $BASE_DIR"
echo "Native resolution: $rawRes_meters m"
echo "Output resolution: $resOut_meters m"
echo "BM correlation kernels: ${bmCKernels[@]}"
echo "MGM correlation kernels: ${mgmCKernels[@]}"
```

This confirms that the correct configuration file has been loaded before running the ASP processing commands.

From this point, the notebook can use the variables defined in the `.sh` file directly inside Bash cells.

---

#### Note on stereo-pair configuration

The number of image pairs should be adjusted depending on the acquisition geometry and the objective of the processing.

For a **standard stereo acquisition**, only one stereo pair is processed:

```text
A and B